In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

C:\Users\HomePC\Desktop\Langchain_foundation\lca-lc-foundations\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[square_root]
)

subagent_2 = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='bb2f5a85-6e44-4af9-96bd-22de3753cda2'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ry0fr56t4', 'function': {'arguments': '{"x":456}', 'name': 'call_subagent_1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 339, 'total_tokens': 356, 'completion_time': 0.020687553, 'completion_tokens_details': None, 'prompt_time': 0.022842856, 'prompt_tokens_details': None, 'queue_time': 0.005998247, 'total_time': 0.043530409}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd75d-ff43-7ac2-b4b1-66b498685f6d-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'ry0fr56t4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i